<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Leo/JobBert_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#JobBert Model

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score

import torch


In [5]:
#Load Data
# Department dataset (labeled)
DEPARTMENT_DATA_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/main/department.csv"
df_department = pd.read_csv(DEPARTMENT_DATA_PATH)

# Profiles dataset (your unlabeled or partially labeled inference set)
PROFILES_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/main/df_profiles_cleansed.csv"
df_profiles = pd.read_csv(PROFILES_PATH)



In [6]:
#Standardize columns + clean
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Standardize common naming differences
    if "position" not in df.columns and "text" in df.columns:
        df = df.rename(columns={"text": "position"})
    if "department" not in df.columns and "label" in df.columns:
        df = df.rename(columns={"label": "department"})

    return df

df_department = standardize_columns(df_department)
df_profiles   = standardize_columns(df_profiles)

# Basic cleaning
df_department = df_department.dropna(subset=["position", "department"]).copy()
df_department["position"] = df_department["position"].astype(str).str.strip()
df_department["department"] = df_department["department"].astype(str).str.strip()

df_profiles = df_profiles.dropna(subset=["position"]).copy()
df_profiles["position"] = df_profiles["position"].astype(str).str.strip()

print("Department rows:", len(df_department))
print("Profiles rows:", len(df_profiles))
print(df_department["department"].value_counts().head(10))


Department rows: 10145
Profiles rows: 2615
department
Marketing                 4295
Sales                     3328
Information Technology    1305
Business Development       620
Project Management         201
Consulting                 167
Administrative              83
Other                       42
Purchasing                  40
Customer Support            33
Name: count, dtype: int64


JobBERT embeddings + Logistic Regression (NO threshold, proper train/test)

In [7]:
#Train/test split on department.csv
le = LabelEncoder()
df_department["label_id"] = le.fit_transform(df_department["department"].values)

dept_train, dept_test = train_test_split(
    df_department,
    test_size=0.2,
    random_state=42,
    stratify=df_department["label_id"]
)

print("Train split:", len(dept_train), "Test split:", len(dept_test))
print("Num labels:", len(le.classes_))


Train split: 8116 Test split: 2029
Num labels: 11


In [8]:
#Encode with SentenceTransformer JobBERT
!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

embedder = SentenceTransformer("TechWolf/JobBERT-v3")

X_train_text = dept_train["position"].tolist()
X_test_text  = dept_test["position"].tolist()

X_train_emb = embedder.encode(
    X_train_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
X_test_emb = embedder.encode(
    X_test_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

y_train = dept_train["label_id"].values
y_test  = dept_test["label_id"].values


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Asym/6235903824_Dense/model.safetensor(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

2_Asym/6235904160_Dense/model.safetensor(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

In [9]:
#Train + evaluate (no thresholding)
clf = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    n_jobs=-1
)
clf.fit(X_train_emb, y_train)

pred_test = clf.predict(X_test_emb)

print("=== BASELINE: Embeddings + LogisticRegression (Department Test) ===")
print("Accuracy:", accuracy_score(y_test, pred_test))
print("Macro F1:", f1_score(y_test, pred_test, average="macro"))
print(classification_report(y_test, pred_test, digits=3, target_names=le.classes_))


=== BASELINE: Embeddings + LogisticRegression (Department Test) ===
Accuracy: 0.9295219319862001
Macro F1: 0.8383296735008721
                        precision    recall  f1-score   support

        Administrative      0.519     0.824     0.636        17
  Business Development      0.776     0.919     0.841       124
            Consulting      0.780     0.970     0.865        33
      Customer Support      0.750     0.857     0.800         7
       Human Resources      0.600     1.000     0.750         6
Information Technology      0.890     0.897     0.893       261
             Marketing      0.984     0.936     0.959       859
                 Other      0.727     1.000     0.842         8
    Project Management      0.636     0.875     0.737        40
            Purchasing      0.889     1.000     0.941         8
                 Sales      0.975     0.938     0.956       666

              accuracy                          0.930      2029
             macro avg      0.775     0.

Fine-tune JobBERT on department train split → evaluate on department test split

In [10]:
#Install + prepare datasets
!pip -q install transformers datasets evaluate accelerate

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate

MODEL_NAME = "TechWolf/JobBERT-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

num_labels = len(le.classes_)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [11]:
train_ds = Dataset.from_pandas(dept_train[["position", "label_id"]].rename(columns={"label_id": "labels"}))
test_ds  = Dataset.from_pandas(dept_test[["position", "label_id"]].rename(columns={"label_id": "labels"}))

def tokenize_fn(batch):
    return tokenizer(
        batch["position"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/8116 [00:00<?, ? examples/s]

Map:   0%|          | 0/2029 [00:00<?, ? examples/s]

In [12]:
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    out = acc_metric.compute(predictions=preds, references=labels)
    out["macro_f1"] = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return out


Fine Tune

In [13]:
model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir="jobbert_dept_finetuned",
    report_to="none",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,   # we evaluate on the held-out department test split
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
eval_out = trainer.evaluate()
eval_out


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at TechWolf/JobBERT-v3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.149600,0.114076,0.989650,0.870265
2,0.065800,0.056302,0.996057,0.991540
3,0.040900,0.046527,0.997043,0.994401


{'eval_loss': 0.04652673378586769,
 'eval_accuracy': 0.9970428782651553,
 'eval_macro_f1': 0.994400706244669,
 'eval_runtime': 5.7292,
 'eval_samples_per_second': 354.152,
 'eval_steps_per_second': 5.585,
 'epoch': 3.0}

In [14]:
#Test split classification report (human-readable labels)
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
pred_ids = logits.argmax(axis=1)

y_true_ids = dept_test["label_id"].values
y_pred_ids = pred_ids

y_true = le.inverse_transform(y_true_ids)
y_pred = le.inverse_transform(y_pred_ids)

print("=== FINETUNED JobBERT (Department Test) ===")
print(classification_report(y_true, y_pred, digits=3))


=== FINETUNED JobBERT (Department Test) ===
                        precision    recall  f1-score   support

        Administrative      0.944     1.000     0.971        17
  Business Development      0.992     1.000     0.996       124
            Consulting      1.000     0.970     0.985        33
      Customer Support      1.000     1.000     1.000         7
       Human Resources      1.000     1.000     1.000         6
Information Technology      0.992     0.985     0.988       261
             Marketing      1.000     0.999     0.999       859
                 Other      1.000     1.000     1.000         8
    Project Management      1.000     1.000     1.000        40
            Purchasing      1.000     1.000     1.000         8
                 Sales      0.997     1.000     0.999       666

              accuracy                          0.997      2029
             macro avg      0.993     0.996     0.994      2029
          weighted avg      0.997     0.997     0.997     

In [20]:
# (A) checking results:

print("dept_train rows:", len(dept_train))
print("dept_test rows:", len(dept_test))
print("train_ds rows:", len(train_ds))
print("test_ds rows:", len(test_ds))


dept_train rows: 8116
dept_test rows: 2029
train_ds rows: 8116
test_ds rows: 2029


In [21]:
#(B) Overlap identischer Titles zwischen Train und Test
train_titles = set(dept_train["position"].str.lower().str.strip())
test_titles  = set(dept_test["position"].str.lower().str.strip())
overlap = train_titles.intersection(test_titles)

print("Unique title overlap:", len(overlap))
print("Test titles seen in train (%):", len(overlap)/len(test_titles)*100)


Unique title overlap: 0
Test titles seen in train (%): 0.0


In [22]:
train_metrics = trainer.evaluate(train_ds)
test_metrics  = trainer.evaluate(test_ds)

print("Train metrics:", train_metrics)
print("Test metrics:", test_metrics)


Train metrics: {'eval_loss': 0.03305862098932266, 'eval_accuracy': 0.9995071463775259, 'eval_macro_f1': 0.9979494838344642, 'eval_runtime': 22.7599, 'eval_samples_per_second': 356.591, 'eval_steps_per_second': 5.58, 'epoch': 3.0}
Test metrics: {'eval_loss': 0.04652673378586769, 'eval_accuracy': 0.9970428782651553, 'eval_macro_f1': 0.994400706244669, 'eval_runtime': 5.7747, 'eval_samples_per_second': 351.359, 'eval_steps_per_second': 5.541, 'epoch': 3.0}


In [23]:
dept_train_shuf = dept_train.copy()
dept_train_shuf["label_id"] = np.random.permutation(dept_train_shuf["label_id"].values)

train_ds_shuf = Dataset.from_pandas(
    dept_train_shuf[["position", "label_id"]].rename(columns={"label_id":"labels"})
).map(tokenize_fn, batched=True)
train_ds_shuf.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

model_sanity = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

trainer_sanity = Trainer(
    model=model_sanity,
    args=training_args,
    train_dataset=train_ds_shuf,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer_sanity.train()
print(trainer_sanity.evaluate())


Map:   0%|          | 0/8116 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at TechWolf/JobBERT-v3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3677283833.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_sanity = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.441000,1.434221,0.423361,0.054080
2,1.415700,1.443594,0.423361,0.054080
3,1.413200,1.472443,0.422868,0.054035


{'eval_loss': 1.434220790863037, 'eval_accuracy': 0.42336126170527355, 'eval_macro_f1': 0.05407957693276253, 'eval_runtime': 5.6873, 'eval_samples_per_second': 356.758, 'eval_steps_per_second': 5.627, 'epoch': 3.0}


In [24]:
import hashlib

def fingerprint_positions(df):
    s = "\n".join(df["position"].astype(str).head(200).tolist())
    return hashlib.md5(s.encode("utf-8")).hexdigest()

print("train fingerprint:", fingerprint_positions(dept_train))
print("test fingerprint :", fingerprint_positions(dept_test))


train fingerprint: 241c711805ef7e8459e34f89e3edb895
test fingerprint : d9b8baf4e371f2af9704890712d6d248


In [25]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import numpy as np

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))


Accuracy: 0.9970428782651553
Balanced accuracy: 0.9957461049859078
Macro F1: 0.994400706244669


3) Profiles inference + thresholding (THIS is where threshold belongs)


In [15]:
#Helper: predict probabilities on any dataframe
from datasets import Dataset

def predict_proba_for_positions(trainer: Trainer, df: pd.DataFrame, text_col="position"):
    ds = Dataset.from_pandas(df[[text_col]].copy())
    ds = ds.map(tokenize_fn, batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    out = trainer.predict(ds)
    logits = out.predictions
    proba = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return proba


In [16]:
#Predict on profiles
proba_p = predict_proba_for_positions(trainer, df_profiles, text_col="position")

pred_idx = proba_p.argmax(axis=1)
pred_conf = proba_p[np.arange(len(proba_p)), pred_idx]
pred_label = le.inverse_transform(pred_idx).astype(object)

df_profiles["pred_department_raw"] = pred_label
df_profiles["pred_conf"] = pred_conf

df_profiles[["position", "pred_department_raw", "pred_conf"]].head(10)


Map:   0%|          | 0/2615 [00:00<?, ? examples/s]

,position,pred_department_raw,pred_conf
0,Prokurist,Information Technology,0.441675
1,CFO,Information Technology,0.351470
2,Betriebswirtin,Other,0.457768
3,Prokuristin,Information Technology,0.464233
4,CFO,Information Technology,0.351470
5,Buchhalterin,Information Technology,0.410324
6,Solutions Architect,Information Technology,0.969944
7,Senior Network Engineer,Information Technology,0.904133
8,Manager of Network Services,Information Technology,0.944445
9,Infrastructure Administrator II,Information Technology,0.974226


In [17]:
#Threshold + optional top-2 margin (recommended)
THRESH = 0.55
USE_MARGIN = True
MARGIN = 0.05

top2 = np.partition(proba_p, -2, axis=1)[:, -2]
margin = pred_conf - top2

pred_final = pred_label.copy()
pred_final[pred_conf < THRESH] = "Other"

if USE_MARGIN:
    unsure = (pred_final != "Other") & (margin < MARGIN)
    pred_final[unsure] = "Other"

df_profiles["pred_department"] = pred_final
df_profiles["pred_margin"] = margin

df_profiles[["position", "pred_department_raw", "pred_conf", "pred_margin", "pred_department"]].head(10)


,position,pred_department_raw,pred_conf,pred_margin,pred_department
0,Prokurist,Information Technology,0.441675,0.252481,Other
1,CFO,Information Technology,0.351470,0.201043,Other
2,Betriebswirtin,Other,0.457768,0.331337,Other
3,Prokuristin,Information Technology,0.464233,0.292436,Other
4,CFO,Information Technology,0.351470,0.201043,Other
5,Buchhalterin,Information Technology,0.410324,0.251950,Other
6,Solutions Architect,Information Technology,0.969944,0.964962,Information Technology
7,Senior Network Engineer,Information Technology,0.904133,0.851752,Information Technology
8,Manager of Network Services,Information Technology,0.944445,0.921413,Information Technology
9,Infrastructure Administrator II,Information Technology,0.974226,0.970221,Information Technology


4) Threshold sweep for Profiles accuracy (only if profiles has ground truth)

In [18]:
def threshold_sweep(y_true, pred_label, pred_conf, thresholds):
    rows = []
    for thr in thresholds:
        pred = pred_label.copy()
        pred[pred_conf < thr] = "Other"
        rows.append({
            "threshold": thr,
            "accuracy": accuracy_score(y_true, pred),
            "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
            "coverage_(not_Other)": float((pred != "Other").mean())
        })
    return pd.DataFrame(rows)

if "department" in df_profiles.columns:
    y_true_p = df_profiles["department"].astype(str).values
    thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

    df_thr = threshold_sweep(
        y_true=y_true_p,
        pred_label=df_profiles["pred_department_raw"].astype(object).values,
        pred_conf=df_profiles["pred_conf"].values,
        thresholds=thresholds
    )
    df_thr
else:
    print("No ground-truth 'department' in df_profiles. Threshold sweep is not possible (accuracy needs labels).")


In [19]:
if "department" in df_profiles.columns:
    print("=== PROFILES CLASSIFICATION REPORT (With Thresholding) ===")
    print(classification_report(df_profiles["department"].astype(str).values,
                                df_profiles["pred_department"].values,
                                digits=3))


=== PROFILES CLASSIFICATION REPORT (With Thresholding) ===
                        precision    recall  f1-score   support

        Administrative      0.311     0.226     0.262        84
  Business Development      0.394     0.474     0.430        78
            Consulting      0.804     0.590     0.680       195
      Customer Support      0.000     0.000     0.000        48
       Human Resources      0.000     0.000     0.000        69
Information Technology      0.429     0.744     0.544       309
             Marketing      0.593     0.549     0.570       133
                 Other      0.677     0.730     0.703      1235
    Project Management      0.820     0.607     0.698       173
            Purchasing      0.000     0.000     0.000        72
                 Sales      0.879     0.795     0.835       219

              accuracy                          0.633      2615
             macro avg      0.446     0.429     0.429      2615
          weighted avg      0.610     0.633

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
